In [5]:
!pip install -U transformers accelerate datasets evaluate sentencepiece protobuf
!pip install -U scikit-learn pandas numpy
!pip install -U torch torchvision torchaudio

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet(
    "hf://datasets/DaniilOr/CoDET-M4/dataset_without_comments.parquet"
)

print("Original shape:", df.shape)
print(df.columns)

df = df[
    [
        "cleaned_code",
        "language",
        "source",
        "model",
        "target",
        "split"
    ]
].copy()

df = df.dropna(subset=["cleaned_code", "language", "target", "split"])

df = df.drop_duplicates(subset=["cleaned_code"])

df = df[df["cleaned_code"].str.len() > 20]

df["label"] = df["target"].map({
    "human": 0,
    "ai": 1
})

print("After cleaning:", df.shape)
print(df["label"].value_counts())
print(df["language"].value_counts())
print(df["split"].value_counts())
print(df["language"].value_counts())

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"].isin(["validation", "valid", "val"])].copy()
test_df = df[df["split"] == "test"].copy()

print("\nOfficial split sizes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

reduced_train_parts = []

languages = ["python", "java", "cpp"]

for lang in languages:
    lang_train = train_df[train_df["language"] == lang]

    human_part = lang_train[lang_train["label"] == 0]
    ai_part = lang_train[lang_train["label"] == 1]

    print(f"\nLanguage: {lang}")
    print("Human:", len(human_part))
    print("AI:", len(ai_part))

    n_samples = min(len(human_part), len(ai_part), 3000)

    sampled_human = human_part.sample(n=n_samples, random_state=42)
    sampled_ai = ai_part.sample(n=n_samples, random_state=42)

    reduced_train_parts.append(sampled_human)
    reduced_train_parts.append(sampled_ai)

reduced_train_df = pd.concat(reduced_train_parts)

reduced_train_df = reduced_train_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nReduced train shape:", reduced_train_df.shape)
print(reduced_train_df["language"].value_counts())
print(reduced_train_df["label"].value_counts())
reduced_train_df.to_csv("train_codebert_reduced.csv", index=False)
val_df.to_csv("val_official.csv", index=False)
test_df.to_csv("test_official.csv", index=False)

print("\nSaved:")
print("train_codebert_reduced.csv")
print("val_official.csv")
print("test_official.csv")

Original shape: (500552, 8)
Index(['code', 'language', 'model', 'split', 'target', 'source', 'features',
       'cleaned_code'],
      dtype='str')
After cleaning: (488291, 7)
label
1    245771
0    242520
Name: count, dtype: int64
language
python    177640
java      171709
cpp       138942
Name: count, dtype: int64
split
train    395477
test      46459
val       46355
Name: count, dtype: int64
language
python    177640
java      171709
cpp       138942
Name: count, dtype: int64

Official split sizes:
Train: (395477, 7)
Validation: (46355, 7)
Test: (46459, 7)

Language: python
Human: 76234
AI: 67611

Language: java
Human: 70526
AI: 68398

Language: cpp
Human: 47777
AI: 64931

Reduced train shape: (18000, 7)
language
python    6000
java      6000
cpp       6000
Name: count, dtype: int64
label
0    9000
1    9000
Name: count, dtype: int64

Saved:
train_codebert_reduced.csv
val_official.csv
test_official.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

train_df = pd.read_csv("train_codebert_reduced.csv")
val_df = pd.read_csv("val_official.csv")
test_df = pd.read_csv("test_official.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

train_df = train_df[["cleaned_code", "label", "language"]]
val_df = val_df[["cleaned_code", "label", "language"]]
test_df = test_df[["cleaned_code", "label", "language"]]

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

model_name = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["cleaned_code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)
train_dataset = train_dataset.remove_columns(["cleaned_code", "language"])
val_dataset = val_dataset.remove_columns(["cleaned_code", "language"])
test_dataset = test_dataset.remove_columns(["cleaned_code", "language"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="./codebert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

trainer.save_model("./best_codebert_model")
tokenizer.save_pretrained("./best_codebert_model")

print("Best model saved.")

(18000, 7)
(46355, 7)
(46459, 7)


Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/46355 [00:00<?, ? examples/s]

Map:   0%|          | 0/46459 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved.


In [ ]:
predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="binary"
)

accuracy = accuracy_score(y_true, y_pred)

print("\n===== TEST RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

print("\n===== CONFUSION MATRIX =====")
print(cm)

codebert_accuracy = accuracy
codebert_precision = precision
codebert_recall = recall
codebert_f1 = f1

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



===== TEST RESULTS =====
Accuracy : 0.9718
Precision: 0.9621
Recall   : 0.9804
F1 Score : 0.9711

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.98      0.96      0.97     23952
           1       0.96      0.98      0.97     22507

    accuracy                           0.97     46459
   macro avg       0.97      0.97      0.97     46459
weighted avg       0.97      0.97      0.97     46459


===== CONFUSION MATRIX =====
[[23082   870]
 [  442 22065]]


In [ ]:
!pip install -q "transformers==4.38.2" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 75.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.0 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
